# Claim Pipeline Demo

Human-verified end-to-end tests for the full `ClaimAgent` pipeline.
Each cell processes one claim folder and renders a rich visual summary:
documents, extracted fields, validation issues, conversation log, and status.

**Prerequisites:** activate the venv and set your API key in `.env` before launching Jupyter.
```bash
source .venv/bin/activate
jupyter notebook demo/claim_pipeline.ipynb
```

**Reply behaviour:** `DemoChatbot` (defined in the setup cell) captures all outbound messages
and returns pre-set replies instead of blocking on `input()`. Set `replies=[...]` per claim
to simulate customer responses, or leave it empty to skip reply rounds.

In [ ]:
import json, sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

from core.llm_adapters import LLMClientFactory
from core.agent import ClaimAgent
from core.chatbot import Chatbot


class DemoChatbot(Chatbot):
    """Chatbot for notebook demos.

    Captures displayed messages so they can be shown in show_claim().
    Returns pre-set replies from the replies list instead of calling input(),
    so the notebook runs end-to-end without blocking.
    """
    def __init__(self, replies: list[str] | None = None):
        self.messages: list[str] = []
        self._replies = list(replies or [])

    def display(self, message: str) -> None:
        self.messages.append(message)

    def ask(self, prompt: str) -> str:
        self.messages.append(prompt)
        reply = self._replies.pop(0) if self._replies else ""
        self.messages.append(f"> {reply if reply else '(skipped)'}")        
        return reply


# Change model_id here or in config/settings.yaml
# LLM = LLMClientFactory.get_client('gemini')
LLM = LLMClientFactory.get_client('qwen_local')
print('LLM ready:', type(LLM).__name__)

In [ ]:
import html as _html
from IPython.display import display, HTML


def show_claim(claim, chatbot: DemoChatbot | None = None):
    """Render a processed Claim as HTML tables in the notebook."""

    STATUS_COLOR = {
        'complete':      '#2d7a2d',
        'incomplete':    '#cc0000',
        'needs_review':  '#b36b00',
    }
    CONF_COLOR = {'high': '#2d7a2d', 'medium': '#b36b00', 'low': '#cc0000'}
    DOC_STATUS_COLOR = {
        'present':   '#2d7a2d',
        'missing':   '#cc0000',
        'duplicate': '#b36b00',
    }
    PARSE_STATUS_COLOR = {
        'complete':    '#2d7a2d',
        'parse_failed': '#cc0000',
        'unprocessed': '#888',
    }

    e = _html.escape

    # --- Header ---
    sc = STATUS_COLOR.get(claim.status, '#888')
    display(HTML(f"""
    <h2 style='font-family:monospace;margin-bottom:4px'>
      {e(claim.claim_id)}
      <span style='font-size:14px;font-weight:normal;color:{sc};margin-left:12px'>
        &#9679; {e(claim.status.upper())}
      </span>
    </h2>
    <p style='font-family:monospace;font-size:12px;color:#888;margin-top:0'>
      uploaded: {e(claim.uploaded_at)} &nbsp;|&nbsp; reply rounds: {claim.reply_count}
    </p>
    """))

    # --- Documents table ---
    display(HTML("<h3 style='font-family:monospace'>Documents</h3>"))
    rows = ''
    for r in claim.doc_table:
        ds_color = DOC_STATUS_COLOR.get(r.doc_status, '#888')
        ps_color = PARSE_STATUS_COLOR.get(r.parse_status, '#888')
        dup_tag = f" ({e(r.duplicate_type)})" if r.duplicate_type else ''
        reason = e(r.status_reason or '')
        rows += (
            f"<tr>"
            f"<td style='padding:4px 10px'>{e(r.file_name)}</td>"
            f"<td style='padding:4px 10px'>{e(r.doc_type)}</td>"
            f"<td style='padding:4px 10px;color:{ds_color}'>{e(r.doc_status)}{dup_tag}</td>"
            f"<td style='padding:4px 10px;color:{ps_color}'>{e(r.parse_status)}</td>"
            f"<td style='padding:4px 10px;font-size:11px;color:#888'>{reason}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <table style='border-collapse:collapse;font-family:monospace;font-size:13px;width:100%'>
      <thead><tr style='background:#f0f0f0'>
        <th style='padding:4px 10px;text-align:left'>file</th>
        <th style='padding:4px 10px;text-align:left'>doc_type</th>
        <th style='padding:4px 10px;text-align:left'>doc_status</th>
        <th style='padding:4px 10px;text-align:left'>parse_status</th>
        <th style='padding:4px 10px;text-align:left'>note</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))

    # --- Extracted fields ---
    if claim.extracted_fields:
        display(HTML("<h3 style='font-family:monospace'>Extracted Fields</h3>"))
        rows = ''
        for name, f in claim.extracted_fields.items():
            cc = CONF_COLOR.get(f.confidence, '#888')
            valid_mark = '&#10003;' if f.valid else '&#10007;'
            valid_color = '#2d7a2d' if f.valid else '#cc0000'
            note = e((f.validation_note or '') + (' | ' + f.confidence_note if f.confidence_note else ''))
            rows += (
                f"<tr>"
                f"<td style='padding:4px 10px'>{e(name)}</td>"
                f"<td style='padding:4px 10px'>{e(str(f.unified_value or ''))}</td>"
                f"<td style='padding:4px 10px;color:{cc}'>{e(f.confidence)}</td>"
                f"<td style='padding:4px 10px;color:{valid_color}'>{valid_mark}</td>"
                f"<td style='padding:4px 10px;color:#888;font-size:11px'>{note}</td>"
                f"</tr>"
            )
        display(HTML(f"""
        <table style='border-collapse:collapse;font-family:monospace;font-size:13px;width:100%'>
          <thead><tr style='background:#f0f0f0'>
            <th style='padding:4px 10px;text-align:left'>field</th>
            <th style='padding:4px 10px;text-align:left'>unified_value</th>
            <th style='padding:4px 10px;text-align:left'>confidence</th>
            <th style='padding:4px 10px;text-align:left'>valid</th>
            <th style='padding:4px 10px;text-align:left'>notes</th>
          </tr></thead>
          <tbody>{rows}</tbody>
        </table>
        """))

    # --- Validation issues ---
    if claim.validation_issues:
        display(HTML("<h3 style='font-family:monospace'>Validation Issues</h3>"))
        rows = ''
        for vi in claim.validation_issues:
            resolved_tag = "<span style='color:#2d7a2d'>[resolved]</span>" if vi.resolved else "<span style='color:#cc0000'>[open]</span>"
            details = '; '.join(f"{e(k)}: {e(v)}" for k, v in (vi.values or {}).items())
            rows += (
                f"<tr>"
                f"<td style='padding:4px 10px'>{e(vi.issue_type)}</td>"
                f"<td style='padding:4px 10px'>{e(vi.field_name or '')}</td>"
                f"<td style='padding:4px 10px;font-size:12px'>{e(vi.description)}</td>"
                f"<td style='padding:4px 10px;font-size:11px;color:#888'>{details}</td>"
                f"<td style='padding:4px 10px'>{resolved_tag}</td>"
                f"</tr>"
            )
        display(HTML(f"""
        <table style='border-collapse:collapse;font-family:monospace;font-size:13px;width:100%'>
          <thead><tr style='background:#f0f0f0'>
            <th style='padding:4px 10px;text-align:left'>type</th>
            <th style='padding:4px 10px;text-align:left'>field</th>
            <th style='padding:4px 10px;text-align:left'>description</th>
            <th style='padding:4px 10px;text-align:left'>values</th>
            <th style='padding:4px 10px;text-align:left'>status</th>
          </tr></thead>
          <tbody>{rows}</tbody>
        </table>
        """))

    # --- Conversation log ---
    if claim.conversation_log:
        display(HTML("<h3 style='font-family:monospace'>Conversation Log</h3>"))
        for cr in claim.conversation_log:
            direction_label = '&#x2192; OUT' if cr.direction == 'outbound' else '&#x2190; IN'
            d_color = '#0055cc' if cr.direction == 'outbound' else '#333'
            msg_html = e(cr.message).replace('\n', '<br>')
            display(HTML(f"""
            <div style='font-family:monospace;font-size:12px;border-left:3px solid {d_color};
                        padding:6px 12px;margin:6px 0;background:#fafafa'>
              <b style='color:{d_color}'>Round {cr.round} {direction_label}</b>
              <span style='color:#888;margin-left:8px'>{e(cr.timestamp)}</span><br>
              <span style='white-space:pre-wrap'>{msg_html}</span>
            </div>
            """))

    # --- Chatbot transcript (displayed messages + replies) ---
    if chatbot and chatbot.messages:
        display(HTML("<h3 style='font-family:monospace'>Agent Output</h3>"))
        for msg in chatbot.messages:
            display(HTML(
                f"<pre style='font-family:monospace;font-size:12px;background:#f5f5f5;"
                f"border:1px solid #ddd;padding:8px;margin:4px 0;white-space:pre-wrap'>"
                f"{e(msg)}</pre>"
            ))


print('show_claim() helper ready')

---
## CLM-001 — All documents present

Three required docs (PDF + PNG). Expect: **complete** with all fields extracted.

In [ ]:
chatbot = DemoChatbot(replies=[])   # no reply needed for a complete claim
agent = ClaimAgent(chatbot=chatbot)
agent.llm_client = LLM

claim = agent.process_claim('../claims/CLM-001')
show_claim(claim, chatbot)

---
## CLM-002 — Missing police_report + customer reply

Only finance_agreement and settlement_breakdown present.
There is a customer_reply.txt in the folder.
Expect: **incomplete** (police_report missing), outbound message, then reply round.

In [ ]:
# Pre-set a simulated customer reply (or leave empty to skip)
REPLY_CLM002 = ""   # e.g. "My VIN is 1HGCM82633A004352 and the date was 2024-01-15."

chatbot = DemoChatbot(replies=[REPLY_CLM002] if REPLY_CLM002 else [])
agent = ClaimAgent(chatbot=chatbot)
agent.llm_client = LLM

claim = agent.process_claim('../claims/CLM-002')
show_claim(claim, chatbot)

---
## CLM-003 — Same-content duplicate

`settlement_breakdown.pdf` and `settlement_breakdown_v2.pdf` have identical bytes.
Expect: **needs_review** (same-content duplicate), duplicate record flagged.

In [ ]:
chatbot = DemoChatbot(replies=[])
agent = ClaimAgent(chatbot=chatbot)
agent.llm_client = LLM

claim = agent.process_claim('../claims/CLM-003')
show_claim(claim, chatbot)

---
## CLM-004 — All images, extra unknown doc

All three required docs are PNGs (scanned images). `tow_receipt.png` is an extra file.
Expect: confidence capped at **medium** for image sources; `tow_receipt` classified as unknown → **needs_review**.

In [ ]:
chatbot = DemoChatbot(replies=[])
agent = ClaimAgent(chatbot=chatbot)
agent.llm_client = LLM

claim = agent.process_claim('../claims/CLM-004')
show_claim(claim, chatbot)

---
## CLM-005 — Missing finance_agreement + customer reply

Only police_report and settlement_breakdown present.
There is a customer_reply.txt.
Expect: **incomplete** (finance_agreement missing), outbound message.

In [ ]:
REPLY_CLM005 = ""   # optional simulated reply

chatbot = DemoChatbot(replies=[REPLY_CLM005] if REPLY_CLM005 else [])
agent = ClaimAgent(chatbot=chatbot)
agent.llm_client = LLM

claim = agent.process_claim('../claims/CLM-005')
show_claim(claim, chatbot)

---
## Batch: process all claims + priority ranking

Runs all five claims and renders the priority table.
Claims already processed above will reuse the `.cache/claim_state.json` on disk
if you load them from JSON; here we reprocess fresh so results are self-contained.

In [ ]:
from core.models import Claim

CLAIMS_DIR = Path('../claims')
claim_dirs = sorted(p for p in CLAIMS_DIR.iterdir() if p.is_dir() and not p.name.startswith('.'))

all_claims = []
for d in claim_dirs:
    print(f'Processing {d.name}...', end=' ')
    try:
        cb = DemoChatbot(replies=[])        # skip all reply rounds in batch mode
        a = ClaimAgent(chatbot=cb)
        a.llm_client = LLM
        c = a.process_claim(str(d))
        all_claims.append(c)
        print(c.status)
    except Exception as exc:
        print(f'ERROR: {exc}')

print(f'\n{len(all_claims)} claims processed.')

In [ ]:
# Build priority ranking from the last agent instance (config is shared)
records = a.prioritize_claims(all_claims)

e = _html.escape
STATUS_COLOR = {'complete': '#2d7a2d', 'incomplete': '#cc0000', 'needs_review': '#b36b00'}

rows = ''
for r in records:
    sc = STATUS_COLOR.get(r.status, '#888')
    express_tag = "<span style='color:#0055cc'>[EXPRESS]</span>" if r.express else ''
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px;text-align:center'><b>{r.priority_rank}</b></td>"
        f"<td style='padding:6px 12px'>{e(r.claim_id)}</td>"
        f"<td style='padding:6px 12px;color:{sc}'>{e(r.status)}</td>"
        f"<td style='padding:6px 12px'>{express_tag}</td>"
        f"<td style='padding:6px 12px;font-size:12px;color:#555'>{e(r.reason)}</td>"
        f"</tr>"
    )

display(HTML(f"""
<h2 style='font-family:monospace'>Priority Ranking</h2>
<table style='border-collapse:collapse;font-family:monospace;font-size:13px;width:100%'>
  <thead><tr style='background:#f0f0f0'>
    <th style='padding:6px 12px'>Rank</th>
    <th style='padding:6px 12px;text-align:left'>Claim</th>
    <th style='padding:6px 12px;text-align:left'>Status</th>
    <th style='padding:6px 12px;text-align:left'>Express</th>
    <th style='padding:6px 12px;text-align:left'>Reason</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))